## Cell 1: Setup and load

In [ ]:
import pandas as pd, numpy as np
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

SQM_TO_SQFT = 10.7639   # documented conversion: 1 square metre = 10.7639 square feet

df = pd.read_csv("../data/raw/transactions-2026-11-04.csv", low_memory=False)
df["INSTANCE_DATE"] = pd.to_datetime(df["INSTANCE_DATE"], errors="coerce")
sales = df[df["GROUP_EN"] == "Sales"].copy()
print("All rows:", len(df), "| Sales rows:", len(sales))

## Cell 2: Category values in Sales (usage, property type, subtype)

In [ ]:
for col in ["USAGE_EN", "PROP_TYPE_EN", "PROP_SB_TYPE_EN"]:
    print(f"\n--- {col} (Sales) ---")
    print(sales[col].value_counts(dropna=False).head(8).to_string())

## Cell 3: Category values in Sales (rooms, parking)

In [ ]:
for col in ["ROOMS_EN", "PARKING"]:
    print(f"\n--- {col} (Sales) ---")
    print(sales[col].value_counts(dropna=False).head(8).to_string())

## Cell 4: Repeated transaction numbers inside Sales

In [ ]:
rep_s = sales[sales["TRANSACTION_NUMBER"].duplicated(keep=False)]
chk = rep_s.groupby("TRANSACTION_NUMBER").agg(
    rows=("TRANS_VALUE", "size"),
    distinct_values=("TRANS_VALUE", "nunique"))

print("Repeated Sales transaction numbers:", len(chk), "| rows inside them:", chk["rows"].sum())
print("Rows per repeated number:", chk["rows"].value_counts().to_dict())
print("% with ONE value on every row:", round((chk["distinct_values"] == 1).mean() * 100, 1))
print("Fully identical rows inside Sales:", sales.duplicated().sum())

diff_cols = [c for c in sales.columns if c != "TRANSACTION_NUMBER"]
differs = rep_s.groupby("TRANSACTION_NUMBER")[diff_cols].nunique(dropna=False).gt(1).sum()
print("\nColumns that differ between rows of the same transaction (out of", len(chk), "transactions):")
if differs.sum() == 0:
    print("None - the rows are identical in every column")
else:
    print(differs[differs > 0].sort_values(ascending=False).to_string())

## Cell 5: Price per sq. ft. for Unit sales (flats and villas)

In [ ]:
units = sales[sales["PROP_TYPE_EN"] == "Unit"].copy()
units["price_per_sqft"] = units["TRANS_VALUE"] / (units["ACTUAL_AREA"] * SQM_TO_SQFT)

print("Unit sales:", len(units), "of", len(sales), "Sales rows")
print(units["price_per_sqft"].describe(percentiles=[.01, .05, .5, .95, .99]).round(0).to_string())

for t in [100, 300]:
    print(f"Units under {t:,} AED/sqft:", (units["price_per_sqft"] < t).sum())
for t in [10000, 50000]:
    print(f"Units over {t:,} AED/sqft:", (units["price_per_sqft"] > t).sum())

## Cell 6: Look at the borderline rows (these decide our outlier thresholds)

In [ ]:
show = ["TRANSACTION_NUMBER", "AREA_EN", "PROP_SB_TYPE_EN", "TRANS_VALUE", "ACTUAL_AREA", "price_per_sqft"]
low = units[(units["price_per_sqft"] >= 100) & (units["price_per_sqft"] < 300)]
high = units[units["price_per_sqft"] > 10000]

print("Between 100 and 300 AED/sqft:", len(low), "(random sample of 8)")
print(low.sample(min(8, len(low)), random_state=42)[show].round(1).to_string())

print("\nAbove 10,000 AED/sqft:", len(high), "(highest 8)")
print(high.sort_values("price_per_sqft", ascending=False).head(8)[show].round(0).to_string())

print("\nAreas with the most rows above 10,000 AED/sqft:")
print(high["AREA_EN"].value_counts().head(5).to_string())

## Cell 7: Area-name variants (same area, different spelling or case)

In [ ]:
key = df["AREA_EN"].str.strip().str.replace(r"\s+", " ", regex=True).str.lower()
variants = df.groupby(key)["AREA_EN"].agg(lambda s: sorted(s.unique()))
print("Unique names:", df["AREA_EN"].nunique(), "| after cleaning case and spaces:", key.nunique())
print(variants[variants.apply(len) > 1].to_string())

## Cell 8: Evidence rows (suspicious values seen earlier)

In [ ]:
cols = ["TRANSACTION_NUMBER", "INSTANCE_DATE", "AREA_EN", "PROP_TYPE_EN",
        "PROP_SB_TYPE_EN", "TRANS_VALUE", "ACTUAL_AREA"]
print("Dubai Marina flat, rows in that transaction:", (df["TRANSACTION_NUMBER"] == "11-31149-2026").sum())
print(df[df["TRANSACTION_NUMBER"] == "11-31149-2026"][cols].to_string())
print("\nLowest 10 Sales values:")
print(sales.nsmallest(10, "TRANS_VALUE")[cols].to_string())